# 02 Feature Engineering

This notebook builds the final feature table for the realistic AML model.

Final feature scope:
- Raw transaction features.
- Leakage-safe historical graph features.
- Rolling-window temporal graph features.
- Advanced graph features
- Node2vec embeddings

In [ ]:
from pathlib import Path
import sys

# Works whether the notebook is launched from repository root or from notebooks/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


import pandas as pd

from src.data.load_data import load_transactions, save_parquet
from src.features.tabular_features import add_basic_transaction_features, make_model_matrix
from src.features.historical_graph_features import add_historical_graph_features
from src.features.rolling_graph_features import add_rolling_graph_features

## 1. Configuration

The final experiments use 2,000,000 rows from HI-Small. Change `NROWS = None` only if you intentionally want to rebuild features using the full HI-Small dataset.

`REBUILD_FEATURES = False` prevents accidentally spending hours rebuilding if `data/processed/hi_small_features.parquet` already exists.

In [ ]:
DATASET_NAME = "HI-Small"
RAW_PATH = RAW_DIR / f"{DATASET_NAME}_Trans.csv"
OUT_PATH = PROCESSED_DIR / "hi_small_features.parquet"

NROWS = 2_000_000
REBUILD_FEATURES = False

print("RAW_PATH:", RAW_PATH)
print("OUT_PATH:", OUT_PATH)
print("NROWS:", NROWS)
print("REBUILD_FEATURES:", REBUILD_FEATURES)

## 2. Build or load processed feature table

In [ ]:
if OUT_PATH.exists() and not REBUILD_FEATURES:
    print("Processed feature table already exists. Loading:", OUT_PATH)
    df_features = pd.read_parquet(OUT_PATH)
else:
    if not RAW_PATH.exists():
        raise FileNotFoundError(
            f"Raw dataset not found: {RAW_PATH}\n"
            "Put HI-Small_Trans.csv inside data/raw/ before running this notebook."
        )

    print("Loading raw data...")
    df = load_transactions(RAW_PATH, nrows=NROWS)

    print("Adding raw transaction features...")
    df_features = add_basic_transaction_features(df)

    print("Adding historical leakage-safe graph features...")
    df_features = add_historical_graph_features(df_features)

    print("Adding rolling-window temporal graph features...")
    df_features = add_rolling_graph_features(df_features, windows=("1h", "24h"))

    print("Saving processed feature table...")
    save_parquet(df_features, OUT_PATH)

print("Processed shape:", df_features.shape)
display(df_features.head())

## 3. Feature inventory

In [ ]:
raw_features = [
    "amount_paid",
    "amount_received",
    "amount_delta",
    "log_amount_paid",
    "log_amount_received",
    "is_cross_bank",
    "is_cross_currency",
    "hour",
    "dayofweek",
    "day",
]

historical_keywords = [
    "_prev",
    "fan_out_score",
    "fan_in_score",
    "pair_repeat_score",
    "sender_amount_ratio",
    "receiver_amount_ratio",
    "graph_activity_score",
]

rolling_keywords = [
    "_1h_",
    "_24h_",
    "time_since_last_tx",
    "rolling_fan",
    "amount_vs_",
]

raw_cols = [c for c in raw_features if c in df_features.columns]
historical_cols = [
    c for c in df_features.columns
    if any(k in c for k in historical_keywords)
]
rolling_cols = [
    c for c in df_features.columns
    if any(k in c for k in rolling_keywords)
]

feature_inventory = pd.DataFrame([
    {"feature_group": "raw_transaction", "count": len(raw_cols), "examples": ", ".join(raw_cols[:8])},
    {"feature_group": "historical_graph", "count": len(historical_cols), "examples": ", ".join(historical_cols[:8])},
    {"feature_group": "rolling_temporal_graph", "count": len(rolling_cols), "examples": ", ".join(rolling_cols[:8])},
])

display(feature_inventory)
feature_inventory.to_csv(TABLES_DIR / "feature_inventory.csv", index=False)
print("Saved:", TABLES_DIR / "feature_inventory.csv")

## 4. Model matrix sanity check

In [ ]:
X_raw, y = make_model_matrix(df_features, include_graph_features=False)
X_graph, y_graph = make_model_matrix(df_features, include_graph_features=True)

matrix_summary = pd.DataFrame([
    {"matrix": "raw", "rows": X_raw.shape[0], "features": X_raw.shape[1], "positive_labels": int(y.sum())},
    {"matrix": "raw_plus_temporal_graph", "rows": X_graph.shape[0], "features": X_graph.shape[1], "positive_labels": int(y_graph.sum())},
])

display(matrix_summary)
matrix_summary.to_csv(TABLES_DIR / "model_matrix_summary.csv", index=False)
print("Saved:", TABLES_DIR / "model_matrix_summary.csv")

## Feature-engineering notes for the report

Use these points in BAB III:
- Historical graph features are computed using previous transactions only.
- Rolling temporal graph features use closed-left time windows, so the current transaction is not included in its own history.
- The final model uses `raw + historical + rolling temporal graph` features.
- This is more realistic than static graph aggregates because it reduces future-information leakage.